## 1) Inspiration

Growing up in the San Francsico Bay Area, I’ve watched as the region became the birthplace of many technologically groundbreaking companies. A huge industry that was a part of this explosion was the reinvention of the taxi. By experience, public transportation isn’t the most accessible so rideshares just made sense. Companies such as Uber and Lyft were founded here nad it has become the testing ground for many autonomous vehicle research companies such as Zoox and Waymo. Not only did I have an internship in high school that was focused on researching autonomous vehicle trends, but now with the knowledge and skills in data science practices, I think this research project is worth exploring once again in a deeper level.

## 2) Stakeholders

My stakeholders of this project will include any Uber or rideshare company interested in understanding these trends to optimize their operations and improve customer satisfaction. This can also be helpful to any local govenrnment agencies who might want this data for future urban planning. Lastly, this can be beneficial to any investors looking at investment opportunities in the rideshare space.



## 3) Task and Metrics

My previous analysis report was on simple trend/data analysis. To take it a step further,  I will be implementing a regression task on the data to predict the fare of an uber ride given certain predictor variables. We will evaluate this model based on RMSE, as we don't want large errors in our model if we are to predict a fare amount. R^2 will also help us identify how well our model can explain the variance. 

## 4) Data

This data is provided by Kaggle Datasets.

Link: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset/data [1]

Our original dataset has 200,000 observations with 8 variables. Each observation represents a single ride.

1. *Key* - a unique identifier for each trip

2. *Fare Amount* - the cost of each trip in usd

3. *Pickup Time* - date and time when the meter was engaged

4. *Passenger Count* - the number of passengers in the vehicle (driver entered value)

5. *Pickup Longitude* - the longitude where the meter was engaged

6. *Pickup Latitude* - the latitude where the meter was engaged

7. *Dropoff Longitude* - the longitude where the meter was disengaged

8. *Dropoff Latitude* - the latitude where the meter was disengaged

Here, there are 7 numeric variables (including our response variable Fare Amount) and 1 datetime variable. 

To clean it, we are removing the "key" variable as it has no significance to our analysis. We will expand the datetime variable into 4 columns (month, day, hour, and day of week). Using the Haversine Formula, we can calculate the distance to include as another variable in the data. Additionally, there is only one observation with missing destination coordinates, therefore we will remove it as it will have little affect and can cause issues in our model if not removed. To handle extreme values that can heavily influence the model, we will limit the fare amount to 200, the distance to 100mi, and the passenger count to 7. In all, we have the variable metrics:

Response: Fare Amount

Predictors: Distance, Pickup longitude, Pickup latitude, Dropoff longitude, Dropoff latitude, Passenger count, month, day, hour, and day of week. 



In [833]:
#| echo: false
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score, confusion_matrix, classification_report

# test train split
from sklearn.model_selection import train_test_split

In [834]:
#| echo: false
data = pd.read_csv('uber.csv')
data = data.drop(columns=['Unnamed: 0', 'key'])
data.dropna(inplace=True)

data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'])
data['month'] = data['pickup_datetime'].dt.month
data['day'] = data['pickup_datetime'].dt.day
data['hour'] = data['pickup_datetime'].dt.hour
data['day_of_week'] = data['pickup_datetime'].dt.dayofweek

data.drop(columns=['pickup_datetime'], inplace=True)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c
data['distance'] = haversine(data['pickup_latitude'], data['pickup_longitude'], data['dropoff_latitude'], data['dropoff_longitude'])

data = data[(data["fare_amount"] > 0) & (data["fare_amount"] < 200)]
data = data[(data["distance"] > 0) & (data["distance"] < 100)]
data = data[(data["passenger_count"] > 0) & (data["passenger_count"] < 7)]

## 5) Prediction

### 1. Linear Regression
Let's begin with a simple linear regression model to predict fare amount with the distance as the only predictor variable. We choose distance because it it logically has the most effect on price. 

In our first model, we can already see how significant the distance predictor is to the model. Our p-value for distance is below a 0.05 threshold indicating that this predictor is statistically significant. The R-squared can also tell us that 71% of the variance can be explained by the model. Now, let's add the other predictors one at a time in the order of: *distance, pickup latitude, pickup longitude, dropoff latitude, dropoff longitude, passenger count, hour, day, month, and day of week*

In [835]:
#| echo: false
X_train, X_test, y_train, y_test = train_test_split(data[["distance"]], data["fare_amount"], test_size=0.2, random_state=1)
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
model = smf.ols(formula='fare_amount ~ distance', data=train).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            fare_amount   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.714
Method:                 Least Squares   F-statistic:                 3.868e+05
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:14:02   Log-Likelihood:            -4.7293e+05
No. Observations:              154550   AIC:                         9.459e+05
Df Residuals:                  154548   BIC:                         9.459e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.9964      0.018    226.178      0.000       3.962       4.031
distance       2.1860      0.004    621.915      0.000       2.179       2.193
==============================================================================
Omnibus:                    86867.235   Durbin-Watson:                   1.997
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        815884567.895
Skew:                          -0.814   Prob(JB):                         0.00
Kurtosis:                     358.943   Cond. No.                         6.89
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

####
### 2. Multi-linear Regression 

When we move into the multi-linear models, they do not improve, but it is possible that the interactions of these terms can make a difference For example, longitude cannot determine any specific location without latitude, and day likely doesn't determine anything without the day of week or month. 

In [842]:
#| echo: false
predictors = ['distance', 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'hour', 'day', 'month', 'day_of_week']

results = []

for i in range(len(predictors)):
    X_train, X_test, y_train, y_test = train_test_split(data[predictors[:i+1]], data["fare_amount"], test_size=0.3, random_state=1)
    model = LinearRegression()
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_rmse = root_mean_squared_error(y_train, train_pred)
    test_rmse = root_mean_squared_error(y_test, test_pred)
    results.append((predictors[:i+1], train_rmse, test_rmse))

model_performances = pd.DataFrame(results, columns=["Predictors", "Train RMSE", "Test RMSE"])
pd.set_option('display.max_colwidth', None)
model_performances

,Predictors,Train RMSE,Test RMSE
0,[distance],5.007371,5.356812
1,"[distance, pickup_latitude]",5.005330,5.357931
2,"[distance, pickup_latitude, pickup_longitude]",5.003474,5.357818
3,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude]",5.002981,5.357742
4,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude]",4.993727,5.331473
5,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count]",4.993206,5.331662
6,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour]",4.993093,5.331708
7,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour, day]",4.993082,5.331733
8,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour, day, month]",4.990941,5.330346
9,"[distance, pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude, passenger_count, hour, day, month, day_of_week]",4.990535,5.329623


### 3. Polynomial Regression with Regularization & Cross-Validation

Using the Scikit-learn machine learning library, we can create a model with the interaction and transformation of these terms together. Additionally, in our previous model, because our train RMSE's were lower than our test RMSE's, it's indicating that it was consistenly overfitting by a small amount. Therefore we can use regularization with cross-validation to shrink the coefficients of less important variables and lessen its impact to prevent overfitting. This specific method of regularization is known as Ridge with cross-validation. 

In [837]:
#| echo: false
X_train, X_test, y_train, y_test = train_test_split(data[predictors], data["fare_amount"], test_size=0.3, random_state=1)

poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

alphas = np.logspace(-3, 3, 200) 
ridge_CV = RidgeCV(alphas=alphas, cv=5, scoring='neg_root_mean_squared_error')
ridge_CV.fit(X_train_scaled, y_train)
best_alpha = ridge_CV.alpha_
train_pred = ridge_CV.predict(X_train_scaled)
test_pred = ridge_CV.predict(X_test_scaled)
train_rmse = root_mean_squared_error(y_train, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)  
print(f"Best alpha: {best_alpha}")
print(f"Train RMSE: {train_rmse}")
print(f"Test RMSE: {test_rmse}")



Best alpha: 1.1895340673703196
Train RMSE: 4.540749949906305
Test RMSE: 4.629173157034097


With Ridge Cross-validation, we were able to achieve a better price model RMSE for both Train and Test predictions. Additionally, our differences in the two RMSEs are smaller in this more complex model, being within a 0.1 margin. Therefore, we have nearly eliminated overfitting while preventing underfitting at the same time. So why did the RMSEs drop overall? 

Shown below, we can see the impact of the top 10 predictors to the model in absolute value out of all 285 predictors. While the distance stays on top as the highest predictor coefficient, close behind are the interactions and transformations of longitude/latitude with distance. This is hardly surprising as distance doesn't have to be uber's only price predictor, as it can depend on pickup and dropoff locations as well. 

In [838]:
#| echo: false
feature_names = poly.get_feature_names_out()
coef_df = pd.DataFrame({"feature": feature_names,"coefficient": ridge_CV.coef_})
coef_df.sort_values(by="coefficient", key=abs,  ascending=False)[:10]


,feature,coefficient
0,distance,-33.172817
99,distance dropoff_longitude^2,31.594559
66,distance^2 pickup_latitude,-24.669595
86,distance pickup_longitude dropoff_longitude,21.274191
67,distance^2 pickup_longitude,-20.843400
13,distance dropoff_latitude,-17.625976
10,distance^2,-15.910831
18,distance month,-15.809674
11,distance pickup_latitude,-15.648572
69,distance^2 dropoff_longitude,-15.225647


On the other end, below are the 5 least important predictors in the model. We can see that passenger count has no impact to the overall price of the model, and this makes sense as carpooling should be promoted by the company for ethical purposes. Along with this, the time metrics also seem to appear frequently, indicating that uber doesn't factor time into their price modeling. 

In [839]:
#| echo: false
coef_df.sort_values(by="coefficient", key=abs,  ascending=True)[:5]

,feature,coefficient
264,passenger_count day_of_week^2,0.006505
203,dropoff_latitude^2 passenger_count,0.007511
258,passenger_count hour day_of_week,0.008519
272,hour month^2,0.011227
279,day month day_of_week,-0.012661


## 6) Inference

Taking the most significant predictors in the model (the distance and longitude/latitude of both pickup and dropoff), I took the linear terms and added distance as an interaction to each spacial variable. While these interaction predictors aren't exactly what we saw to have the the highest imapct on the model, by looking at the top 10, we see that distance is in every predictor, therefore it would make sense to have this as the main interaction term. The equation is as follows: 

*fare_amount = distance + pickup_latitude + pickup_longitude + dropoff_latitude + dropoff_longitude + distance(pickup_latitude + pickup_longitude + dropoff_latitude + dropoff_longitude)*

In [843]:
model = smf.ols(formula='fare_amount ~ distance + pickup_latitude + pickup_longitude + dropoff_latitude + dropoff_longitude +' \
'distance*pickup_latitude + distance*pickup_longitude + distance*dropoff_latitude + distance*dropoff_longitude', data=data).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            fare_amount   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                 6.685e+04
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        13:39:55   Log-Likelihood:            -5.7529e+05
No. Observations:              193188   AIC:                         1.151e+06
Df Residuals:                  193178   BIC:                         1.151e+06
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     75.9023      1.285     59.081      0.000      73.384      78.420
distance                     -73.9684      0.462   -160.132      0.000     -74.874     -73.063
pickup_latitude               -7.6454      0.531    -14.410      0.000      -8.685      -6.606
pickup_longitude              27.3837      0.417     65.660      0.000      26.566      28.201
dropoff_latitude               9.7909      0.532     18.393      0.000       8.748      10.834
dropoff_longitude            -25.2222      0.415    -60.739      0.000     -26.036     -24.408
distance:pickup_latitude       0.2129      0.015     13.979      0.000       0.183       0.243
distance:pickup_longitude     -2.2565      0.022   -103.585      0.000      -2.299      -2.214
distance:dropoff_latitude     -2.5109      0.023   -110.415      0.000      -2.555      -2.466
distance:dropoff_longitude    -0.0417      0.018     -2.329      0.020      -0.077      -0.007
==============================================================================
Omnibus:                   246577.101   Durbin-Watson:                   2.003
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        183395800.206
Skew:                           6.528   Prob(JB):                         0.00
Kurtosis:                     153.376   Cond. No.                     7.25e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 7.25e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### **Effects of the Predictors on Fare Amount**
#### 1. Distance
Distance alone continues to have the most effect on the fare amount with a coefficient of -73.9684. While this coefficient may seem extreme, this really means that distance doesn't have a constant effect on the fare amount. Instead, the true marginal effect of distance depends on the pickup and dropoff locations as well. 

#### 2. The 4 spacial predictors (latitude and longitude) 
For these predictors, they all have similar linear effects, but  in different directions. Each of these predictors are highly significant because they each fall below our p-value threshold of 0.05.

Pickup Latitude (–7.65)

- A one‑degree increase in pickup latitude (moving north) reduces fare by about $7.65, holding all else constant. This reflects spatial patterns in the dataset—northern pickups may be closer to common destinations.

Pickup Longitude (+27.38)

- A one‑degree increase in pickup longitude (moving east) increases fare by $27.38. Eastern pickups may be farther from typical dropoff clusters.

Dropoff Latitude (+9.79)

- A one‑degree increase in dropoff latitude (moving north) increases fare by $9.79, suggesting northern dropoffs tend to be farther or more expensive routes.

Dropoff Longitude (–25.22)

- A one‑degree increase in dropoff longitude (moving east) decreases fare by $25.22, indicating that eastern dropoffs tend to be cheaper.

#### 3. The 4 distance x spacial predictors 
These predictors take the product of distance and each predictor separately. It essentially means that the cost of an extra mile depends on where the trip happens.

distance × pickup_latitude (+0.213)
- For more northern pickups, each additional mile increases fare by an extra $0.21. Distance becomes more expensive in northern regions.

distance × pickup_longitude (–2.256)
- For more eastern pickups, each additional mile becomes $2.26 cheaper. Distance is less costly in the east.

distance × dropoff_latitude (–2.511)
- For northern dropoffs, each additional mile becomes $2.51 cheaper. This suggests northern destinations are closer to major routes.

distance × dropoff_longitude (–0.042)
- A much small negative effect: eastern dropoffs slightly reduce the marginal cost of distance.

While the interaction predictors don't have as much of an effect to the model, we can see that they are still statistically significant with p-values below the 0.05 threshold. The only borderline term is distance × dropoff_longitude, but even this is significant at the 2% level. This means that the direction and magnitude of the estimated effects are highly reliable.

### **Variation Explained**
R-squared (the amount of variation explained in the model) will always increase or stay constant with more predictors added. Our R-squared falls at 75.7%, a 4% increase from our previous linear model with only distance as the predictor. This means that we were able to explain more variation, and even a 4% change is substantial enought to show relevancy to our price prediction.

## 7) Recommendation to Stakeholders

For stakeholders, although our models show statistical significance, I wouldn't recommend using these models for operational or financial decision making. Instead this analysis should be used as for inference and understanding of certain factors and their impact on the fare amount. Clearly, and most intuitively, distance matters the most as it's directly linked to gas mileage, gas pricing, and time. But one one the factors that I didn't expect to have as much impact as it did was spacial coordinates. This makes sense as the price of living varies drastically based on location, and this directly affects the price of our ride fare.

Every analysis has its limitations, and there are certainly a few factors that we couldn't consider here. For example, the mpg(miles/gallon) can change between cars, gas prices can vary by the day, and demand for rides can fluctuate throughout the hour. It's likely that Uber considers these factors when determining it's price, but they can also determine them based on their competitors such as lyft. Economic principles tell us that in perfect competition, it is a zero economic profit market and firms must match prices at the minimum price of operation. 

Uber likely determines their prices this way as well, which will require a different analysis. To do this, we can take the fare amounts of both Uber and Lyft and see how reactive/correlated their prices are at certain times and distances. But unfortunately, we are heavily limited on this data to compare an exact location with another and the exact time unless we collect the data directly from their apps and record it simultaneously with each other. 

## 8) Conclusion

According to Uber, the upfront rider prices are based on the estimated length and duration of the trip, but it may change "due to a number of circumstances, which may include adding stops, updating your destination, significant changes to the route or duration of the trip, or you pass through a toll that was not factored into your upfront price."[2] They don't explicitly state how their model predicts fares, but we can safely assume that they are trained across far more complex, dynamic, and data-rich models. 

In conclusion, our models were able to determinine that the main factors when pricing uber rides are distance and location. While this is not precisely how uber prices their rides, we can use this to understand spatial patterns in pricing. The models reveal that fare structure varies across regions and that although distance interacts weakly with the indiividual longitude and latitude of the pickup and dropoff coordinates, they continue to have statistical significance to our alternative hypothesis. This is valuable for identifying geographic trends, inefficiencies, or areas with systematically higher or lower fares.

While a test data's RMSE of ~4.6 sounds good, it still has a meaningful amount of prediction error that limits how the model can be used. This can be the difference between a negative profit for the company and a loss of business with a customer to another competitor that has a lower fare price. Overall, this analysis provides meaningful insight into how distance and geography shape fare patterns, but its predictive limitations mean it should inform strategic understanding rather than operational pricing decisions.

## References

[1] Kaggle Datasets. “Uber Fares Dataset.” Www.kaggle.com, 2022, www.kaggle.com/datasets/yasserh/uber-fares-dataset/data.

[2] Uber. “Ride Prices and Rates - How It Works.” Uber, www.uber.com/us/en/ride/how-it-works/upfront-pricing/.